In [ ]:
"""Train a graph classification explainee and wrap it with a configurable `torch_geometric.explain.Explainer`."""
import torch
from typing import Any, Callable, Dict, Optional, Tuple, Union

from revisions.new_src.dataAdapter import load_dataset
from revisions.new_src.explainee import fit_explainee
from torch_geometric.data import Batch
from torch_geometric.explain import CaptumExplainer, Explainer
from torch_geometric.explain.algorithm import GNNExplainer, PGExplainer

AlgorithmSpec = Union[str, Callable[..., Any]]
ExplainerBundle = Tuple[torch.nn.Module, Explainer, Any, Dict[str, Any]]

_EXPLAINER_REGISTRY: Dict[str, Callable[..., Any]] = {
    'gnnexplainer': GNNExplainer,
    'gnn': GNNExplainer,
    'pgexplainer': PGExplainer,
    'pg': PGExplainer,
    'saliency': lambda **kwargs: CaptumExplainer(algorithm='Saliency', **kwargs),
    'integrated_gradients': lambda **kwargs: CaptumExplainer(algorithm='IntegratedGradients', **kwargs),
    'ig': lambda **kwargs: CaptumExplainer(algorithm='IntegratedGradients', **kwargs),
}

class ExplaineeWrapper(torch.nn.Module):
    def __init__(self, base_model: torch.nn.Module):
        super().__init__()
        self.base_model = base_model

    def forward(self, x, edge_index, edge_attr=None, batch=None):
        batch_data = Batch(x=x, edge_index=edge_index, batch=batch)
        kwargs = {}
        if edge_attr is not None:
            if edge_attr.dim() == 1:
                kwargs['edge_weight'] = edge_attr
            else:
                batch_data.edge_attr = edge_attr
        output = self.base_model(batch=batch_data, **kwargs)
        if isinstance(output, dict):
            if 'probs' in output:
                return output['probs']
            if 'logits' in output:
                return output['logits'].softmax(dim=-1)
        return output

def _infer_num_classes(dataset) -> int:
    if hasattr(dataset, 'GRAPH_CLS') and dataset.GRAPH_CLS:
        return len(dataset.GRAPH_CLS)
    if hasattr(dataset, 'num_classes') and dataset.num_classes is not None:
        return int(dataset.num_classes)
    labels = []
    for data in dataset:
        y = getattr(data, 'y', None)
        if y is None:
            continue
        y = y.view(-1)
        if y.numel():
            labels.append(int(y[0].item()))
    if not labels:
        raise ValueError('Unable to infer the number of classes from the dataset')
    return max(labels) + 1

def _resolve_algorithm(name: AlgorithmSpec, kwargs: Optional[Dict[str, Any]] = None):
    kwargs = dict(kwargs or {})
    if callable(name):
        return name(**kwargs)
    key = str(name).lower()
    if key not in _EXPLAINER_REGISTRY:
        raise KeyError(f"Unknown explainer algorithm '{name}'. Registered: {list(_EXPLAINER_REGISTRY)}")
    factory = _EXPLAINER_REGISTRY[key]
    return factory(**kwargs)

def train_explainee_and_build_explainer(
    dataset_name: str,
    *,
    data_root: str = 'data',
    train_kwargs: Optional[Dict[str, Any]] = None,
    algorithm: AlgorithmSpec = 'gnnexplainer',
    algorithm_kwargs: Optional[Dict[str, Any]] = None,
    explainer_kwargs: Optional[Dict[str, Any]] = None,
    device: Optional[Union[str, torch.device]] = None,
) -> ExplainerBundle:
    device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    dataset = load_dataset(dataset_name, root=data_root)

    fit_args = dict(epochs=30, batch_size=64, lr=5e-4)
    if train_kwargs:
        fit_args.update(train_kwargs)
    fit_args.setdefault('device', device)

    explainee, training_stats = fit_explainee(dataset_name, root=data_root, **fit_args)
    explainee = explainee.to(device)
    explainee.eval()

    num_classes = _infer_num_classes(dataset)
    node_mask_type = 'attributes' if getattr(dataset[0], 'x', None) is not None else 'object'
    edge_mask_type = 'object'

    algorithm_instance = _resolve_algorithm(algorithm, algorithm_kwargs)
    model_config = dict(
        mode='binary_classification' if num_classes == 2 else 'multiclass_classification',
        task_level='graph',
        return_type='probs',
    )
    explainer_defaults = dict(
        explanation_type='phenomenon',
        node_mask_type=node_mask_type,
        edge_mask_type=edge_mask_type,
        model_config=model_config,
    )
    if explainer_kwargs:
        explainer_defaults.update(explainer_kwargs)

    wrapper = ExplaineeWrapper(explainee).to(device)
    explainer = Explainer(model=wrapper, algorithm=algorithm_instance, **explainer_defaults)

    return explainee, explainer, dataset, training_stats

DATASET_NAME = 'MUTAG'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

explainee, explainer, dataset, training_log = train_explainee_and_build_explainer(
    DATASET_NAME,
    device=DEVICE,
    algorithm='gnnexplainer',
    algorithm_kwargs=dict(epochs=200),
)
print(f'Explainee trained on {DATASET_NAME} – final stats: {training_log}')
print(f'Using explainer: {explainer.algorithm.__class__.__name__}')


In [ ]:
"""Aggregate instance-level explanations into reusable motif batches."""
import torch
from itertools import chain
from collections import Counter, defaultdict
from typing import Any, Callable, Dict, List, MutableMapping, Optional, Sequence, Tuple, Union

from torch_geometric.data import Batch, Data
from torch_geometric.explain import Explainer
from torch_geometric.utils import subgraph

from revisions.new_src.agg_instance import extract_motif_components, wl_hash

RawMotifs = Dict[int, List[Data]]
MotifBatches = Dict[int, List[Batch]]
AggregationStrategy = Callable[[Sequence[Data]], List[Batch]]

def _infer_device(device, explainer: Explainer, explainee: torch.nn.Module) -> torch.device:
    if device is not None:
        return torch.device(device)
    model = getattr(explainer, 'model', None)
    if model is not None:
        param = next(model.parameters(), None)
        if param is not None:
            return param.device
    param = next(explainee.parameters(), None)
    if param is not None:
        return param.device
    return torch.device('cpu')

def _reindex_motif(motif: Data) -> Data:
    motif = motif.clone()
    edge_index = motif.edge_index
    if edge_index.numel() == 0:
        motif.num_nodes = int(getattr(motif, 'num_nodes', 0) or 0)
        return motif
    max_index = int(edge_index.max().item())
    num_nodes = int(motif.num_nodes) if getattr(motif, 'num_nodes', None) is not None else None
    needs_relabel = num_nodes is None or max_index >= num_nodes
    if not needs_relabel:
        return motif
    node_ids = torch.unique(edge_index)
    node_ids, _ = torch.sort(node_ids)
    new_edge_index, _ = subgraph(node_ids, edge_index, relabel_nodes=True)
    motif.edge_index = new_edge_index
    motif.num_nodes = int(node_ids.numel())
    if motif.x is not None:
        motif.x = motif.x[node_ids]
    return motif

def gather_instance_motifs(
    graphs: Sequence[Data],
    *,
    explainer: Explainer,
    explainee: torch.nn.Module,
    mask_top_p: float = 0.2,
    device: Optional[Union[str, torch.device]] = None,
    use_ground_truth: bool = True,
) -> RawMotifs:
    if not 0.0 < mask_top_p <= 1.0:
        raise ValueError('mask_top_p must lie in (0, 1]')

    device = _infer_device(device, explainer, explainee)
    explainee = explainee.to(device)
    explainee.eval()
    model = getattr(explainer, 'model', None)
    if model is not None:
        model.to(device)
        model.eval()

    motifs: MutableMapping[int, List[Data]] = defaultdict(list)

    for data in graphs:
        working = data.to(device)
        batch_vec = torch.zeros(working.num_nodes, dtype=torch.long, device=device)
        edge_attr = getattr(working, 'edge_attr', None)
        if edge_attr is None:
            edge_attr = getattr(working, 'edge_weight', None)

        target = None
        if use_ground_truth and hasattr(working, 'y'):
            y = working.y.view(-1)
            if y.numel():
                target = int(y[0].item())

        explanation_kwargs = dict(
            x=working.x,
            edge_index=working.edge_index,
            edge_attr=edge_attr,
            batch=batch_vec,
        )
        if target is not None:
            explanation_kwargs['target'] = target
        explanation = explainer(**explanation_kwargs)
        edge_mask = explanation.edge_mask
        if edge_mask is None:
            node_mask = getattr(explanation, 'node_mask', None)
            if node_mask is None:
                raise RuntimeError('Explainer did not provide edge or node masks')
            edge_mask = 0.5 * (node_mask[working.edge_index[0]] + node_mask[working.edge_index[1]])

        edge_mask = edge_mask.detach().float().cpu()
        working_cpu = working.to('cpu')
        motif_subgraphs = [
            _reindex_motif(motif)
            for motif in extract_motif_components(working_cpu, edge_mask, p=mask_top_p)
        ]
        if not motif_subgraphs:
            continue

        cls_idx = target
        if cls_idx is None:
            batch_graph = Batch.from_data_list([working]).to(device)
            prediction = explainee(batch=batch_graph)
            if isinstance(prediction, dict):
                probs = prediction.get('probs')
                if probs is None and 'logits' in prediction:
                    probs = prediction['logits'].softmax(dim=-1)
            else:
                probs = prediction
            cls_idx = int(probs.argmax(dim=-1).item())

        motifs[cls_idx].extend(motif_subgraphs)

    return {cls: list(subgraphs) for cls, subgraphs in motifs.items()}

def _aggregate_wl_topk(motifs: Sequence[Data], *, top_k: int = 5, wl_hops: int = 2) -> List[Batch]:
    if not motifs:
        return []
    counts: Counter = Counter()
    grouped: Dict[int, List[Data]] = defaultdict(list)
    for motif in motifs:
        key = wl_hash(motif, hops=wl_hops)
        grouped[key].append(motif)
        counts[key] += 1
    top = counts.most_common(top_k)
    return [Batch.from_data_list(grouped[key]) for key, _ in top]

def _aggregate_by_size(motifs: Sequence[Data], *, top_k: int = 5) -> List[Batch]:
    if not motifs:
        return []
    grouped: Dict[Tuple[int, int], List[Data]] = defaultdict(list)
    counts: Counter = Counter()
    for motif in motifs:
        if getattr(motif, 'num_nodes', None) is not None and motif.num_nodes > 0:
            num_nodes = int(motif.num_nodes)
        elif getattr(motif, 'x', None) is not None:
            num_nodes = int(motif.x.size(0))
        else:
            num_nodes = int(motif.edge_index.unique().numel())
        num_edges = int(motif.edge_index.size(1))
        key = (num_nodes, num_edges)
        grouped[key].append(motif)
        counts[key] += 1
    ordered = sorted(counts.items(), key=lambda item: (-item[1], item[0]))[:top_k]
    return [Batch.from_data_list(grouped[key]) for key, _ in ordered]

_AGGREGATION_REGISTRY: Dict[str, Callable[..., List[Batch]]] = {
    'wl_topk': _aggregate_wl_topk,
    'weisfeiler_lehman': _aggregate_wl_topk,
    'size_bucket': _aggregate_by_size,
}

def aggregate_common_motifs(
    motifs: RawMotifs,
    *,
    strategy: Union[str, AggregationStrategy] = 'wl_topk',
    strategy_kwargs: Optional[Dict[str, Any]] = None,
) -> MotifBatches:
    if callable(strategy):
        aggregator = strategy
    else:
        key = str(strategy).lower()
        if key not in _AGGREGATION_REGISTRY:
            raise KeyError(
                f"Unknown aggregation strategy '{strategy}'. Registered: {list(_AGGREGATION_REGISTRY)}"
            )
        aggregator = _AGGREGATION_REGISTRY[key]
    kwargs = dict(strategy_kwargs or {})
    return {
        cls_idx: aggregator(class_motifs, **kwargs)
        for cls_idx, class_motifs in motifs.items()
    }

raw_motifs = gather_instance_motifs(
    dataset,
    explainer=explainer,
    explainee=explainee,
    mask_top_p=0.15,
    device=DEVICE,
)
common_motif_batches = aggregate_common_motifs(
    raw_motifs,
    strategy='wl_topk',
    strategy_kwargs=dict(top_k=5, wl_hops=2),
)

print({cls: [batch.num_graphs for batch in batches] for cls, batches in common_motif_batches.items()})

motifs_as_graphs = {
    cls: list(chain.from_iterable(batch.to_data_list() for batch in batches))
    for cls, batches in common_motif_batches.items()
}


In [ ]:
"""Apply the quantitative evaluation pipeline defined in ``new_src/eval.py``."""
from collections import defaultdict
from typing import Dict, List

from torch_geometric.data import Data

from revisions.new_src.eval import eval_summary
from revisions.new_src.graph_level_dist import mcs_soft_graph_dist

observed_by_class: Dict[int, List[Data]] = defaultdict(list)
for graph in dataset:
    label = graph.y.view(-1)[0].item() if graph.y.numel() else 0
    observed_by_class[int(label)].append(graph)

if len(observed_by_class) < 2:
    raise RuntimeError('eval_summary currently expects at least two classes in the dataset.')

class0, class1 = sorted(observed_by_class.keys())[:2]
obs_graphs_0 = observed_by_class[class0]
obs_graphs_1 = observed_by_class[class1]

gen_graphs_0 = motifs_as_graphs.get(class0, obs_graphs_0)
gen_graphs_1 = motifs_as_graphs.get(class1, obs_graphs_1)

dist_to_0 = mcs_soft_graph_dist(obs_graphs_0, temperature=0.1)
dist_to_1 = mcs_soft_graph_dist(obs_graphs_1, temperature=0.1)

summary_score = eval_summary(
    explainee,
    gen_graphs_0,
    gen_graphs_1,
    obs_graphs_0,
    obs_graphs_1,
    dist_to_0,
    dist_to_1,
)
print(f'Composite evaluation score: {summary_score:.4f}')


In [ ]:
"""Visualise generated motifs alongside observed graphs using ``eval_plot``."""
from revisions.new_src.graph_level_dist import mcs_soft_graph_dist
from revisions.new_src.utils import eval_plot

union_reference = obs_graphs_0 + obs_graphs_1
shared_distance = mcs_soft_graph_dist(union_reference, temperature=0.1)

class_labels = getattr(dataset, 'GRAPH_CLS', {})
label_names = [
    class_labels.get(class0, f'Class {class0}'),
    class_labels.get(class1, f'Class {class1}'),
]

max_pairs_to_plot = 3
plot_gen_0 = gen_graphs_0[:max_pairs_to_plot]
plot_gen_1 = gen_graphs_1[:max_pairs_to_plot]
plot_obs_0 = obs_graphs_0[:max_pairs_to_plot]
plot_obs_1 = obs_graphs_1[:max_pairs_to_plot]

eval_plot(
    explainee,
    gen_graphs_0=plot_gen_0,
    gen_graphs_1=plot_gen_1,
    obs_graphs_0=plot_obs_0,
    obs_graphs_1=plot_obs_1,
    ged_model=shared_distance,
    dataset=dataset,
    max_pairs=max_pairs_to_plot,
    device=DEVICE,
    class_labels=label_names,
)
